## Imports

In [ ]:
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set global font family and size
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif", "serif"]
mpl.rcParams["font.size"] = 12  # Adjust per Springer guidelines
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 13
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["legend.fontsize"] = 11

# Optional: PDF/LaTeX text rendering for enhanced output
mpl.rcParams["text.usetex"] = False


## Compare Models

### SOTA

In [ ]:
# micro f1: 0.983647, macro f1: 0.982681, Accuracy: 0.983647
# https://ieeexplore.ieee.org/document/10800870
from sklearn.metrics import f1_score
import numpy as np

cm = np.array([[16627, 286], [362, 19629]])
y_true = np.concatenate([np.zeros(cm[0, 0] + cm[0, 1]), np.ones(cm[1, 0] + cm[1, 1])])
y_pred = np.concatenate(
    [np.zeros(cm[0, 0]), np.ones(cm[0, 1]), np.zeros(cm[1, 0]), np.ones(cm[1, 1])]
)
micro_f1 = f1_score(y_true, y_pred, average="micro")
macro_f1 = f1_score(y_true, y_pred, average="macro")
accuracy = (y_true == y_pred).mean()
print(f"micro f1: {micro_f1:.6f}, macro f1: {macro_f1:.6f}, Accuracy: {accuracy:.6f}")

# https://ieeexplore.ieee.org/document/10711785
cm = np.array(
    [
        [18478, 275 + 11 + 36],
        [6 + 349 + 122 + 157, 9339 + 8730 + 2108 + 134 + 15 + 1445],
    ]
)
y_true = np.concatenate([np.zeros(cm[0, 0] + cm[0, 1]), np.ones(cm[1, 0] + cm[1, 1])])
y_pred = np.concatenate(
    [
        np.zeros(cm[0, 0]),
        np.ones(cm[0, 1]),
        np.zeros(cm[1, 0]),
        np.ones(cm[1, 1]),
    ]
)
micro_f1 = f1_score(y_true, y_pred, average="micro")
macro_f1 = f1_score(y_true, y_pred, average="macro")
accuracy = (y_true == y_pred).mean()
print(f"micro f1: {micro_f1:.6f}, macro f1: {macro_f1:.6f}, Accuracy: {accuracy:.6f}")


### Ours

In [ ]:

csv_dir = Path(
    r"rosaidresults\image_classification\summary_model_evaluation.csv"
)


def change_data_type(x):
    if "image" in x:
        return "ORIGINAL"
    else:
        x = x.split("_")[-1]
        if x == "frequency":
            return "PAYLOAD FREQUENCY"
    return x


selected_model = [
    # "ROSIDS23_blockcnn2d__nosampling_binary_20260102",
    "blockcnn2d_nosampling_binary_20260124",
    "mobilenet_v3_large_nosampling_binary_20260124",
    "resnet18_nosampling_binary_20260124",
    "mobilenet_v3_large_nosampling_binary_20260124",
    "resnet18_nosampling_binary_20260123",
    "blockcnn2d_nosampling_binary_20260123",
    "mobilenet_v3_large_nosampling_binary_20260123",
]
df = pd.read_csv(csv_dir)
df["data_type"] = df.data_type.apply(change_data_type)
df = df.query(
    "filtered_columns==False and model_name_full in @selected_model and 'FILTERED' not in data_type"
)
df.sort_values(by=["macro_f1"], ascending=False)


In [ ]:
pd.read_csv(csv_dir)

In [ ]:
plt.figure(figsize=(6, 6))
ax = sns.barplot(data=df, x="data_type", y="macro_f1", hue="model_name")

for bars in ax.containers:
    ax.bar_label(bars, fmt="%.3f", padding=3)

# Add grid lines
ax.grid(True, alpha=0.3, linestyle="--")

# plt.title("Macro F1 by Data Type and Model")
plt.ylabel("Macro F1 Score")
plt.xlabel("Session Image Type")
plt.legend(title="Model Name", loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
df

In [ ]:
df = pd.read_csv(
    r"rosaidresults\evaluation\image_classification\final_evaluation_results.csv"
)
df["is_normalized"] = df.model_name.str.lower().apply(lambda x: "normalized" in x)

# make dummy data

df

In [ ]:
import numpy as np

# plot bar plot between only for ROSIDS23 dataset
rdf = df.query("data_type=='ROSIDS23'")
# Get unique data types and F1 score columns
data_types = ["ROSIDS23"]
f1_columns = ["weighted_f1_score", "micro_f1_score", "macro_f1_score"]

# Create subplots - rows for data types, columns for F1 scores
fig, axes = plt.subplots(
    len(data_types), len(f1_columns), figsize=(15, 5 * len(data_types))
)

# ROBUST axes handling for ANY number of rows/columns
if len(data_types) == 1:
    axes = np.atleast_2d(axes) if hasattr(axes, "__iter__") else np.array([[axes]])
elif len(f1_columns) == 1:
    axes = np.array([axes]).T

# Colors for normal vs normalized
colors = ["#1f77b4", "#ff7f0e"]  # Blue for normal, orange for normalized
labels = ["Normal", "Normalized"]

# Bar width and positions
bar_width = 0.35
x_positions = np.arange(len(labels))  # [0, 1]

# Iterate through data types and F1 columns
for i, data_type in enumerate(data_types):
    data_type_df = df[df["data_type"] == data_type]

    for j, f1_col in enumerate(f1_columns):
        ax = (
            axes[i, j]
            if axes.ndim == 2
            else axes[0, j]
            if len(data_types) == 1
            else axes[i]
        )

        # Get scores for normal and normalized models
        scores = []
        for is_norm in [False, True]:
            filtered_data = data_type_df[data_type_df["is_normalized"] == is_norm]
            if not filtered_data.empty:
                score = filtered_data[f1_col].max()
            else:
                score = 0
            scores.append(score)

        # Create bars
        bars = ax.bar(
            x_positions,
            scores,
            bar_width,
            color=colors,
            alpha=0.8,
            edgecolor="black",
            linewidth=0.8,
        )

        # SAFE value labels on bars
        max_score = max(scores)
        y_offset = max(0.01, max_score * 0.02)  # Adaptive offset

        for k, (bar, score) in enumerate(zip(bars, scores)):
            if score > 0:
                height = bar.get_height()
                ax.text(
                    bar.get_x() + bar.get_width() / 2.0,
                    height + y_offset * 0.5,
                    f"{score:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=11,
                    fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.9),
                )

        # Customize subplot
        if i == 0:  # Column titles (top row only)
            ax.set_title(
                f"{f1_col.replace('_', ' ').title()}",
                fontweight="bold",
                pad=15,
                fontsize=14,
            )

        if j == 0:  # Row labels (first column only)
            ax.text(
                -0.12,
                0.5,
                data_type,
                rotation=90,
                va="center",
                ha="center",
                transform=ax.transAxes,
                fontweight="bold",
                fontsize=13,
            )

        # X-axis only on bottom row
        if i == len(data_types) - 1:
            ax.set_xticks(x_positions)
            ax.set_xticklabels(labels, fontsize=12, fontweight="bold")

        ax.set_ylabel("F1 Score" if j == 0 else "", fontsize=12, fontweight="bold")
        ax.set_ylim(0, 1.08)
        ax.grid(True, alpha=0.3, axis="y")

        # Clean spines
        for spine in ax.spines.values():
            spine.set(linewidth=0.8)

# Overall title
fig.suptitle(
    "Model Performance: Normal vs Normalized (All Data Types)",
    fontsize=16,
    fontweight="bold",
    y=0.98,
)

plt.tight_layout()
plt.subplots_adjust(top=0.92, left=0.12, right=0.98)
plt.savefig("f1_comparison_all.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Get unique data types and F1 score columns
data_types = df["data_type"].unique()
f1_columns = ["weighted_f1_score", "micro_f1_score", "macro_f1_score"]

# Create subplots - rows for data types, columns for F1 scores
fig, axes = plt.subplots(
    len(data_types), len(f1_columns), figsize=(15, 5 * len(data_types))
)

# Ensure axes is 2D array
if len(data_types) == 1:
    axes = axes.reshape(1, -1)
elif len(f1_columns) == 1:
    axes = axes.reshape(-1, 1)

# Colors for normal vs normalized
colors = ["#1f77b4", "#ff7f0e"]  # Blue for normal, orange for normalized
labels = ["Normal", "Normalized"]

# Bar width and positions
bar_width = 0.35
x_positions = [0, 1]  # Two positions for normal and normalized

# Create plots
for i, data_type in enumerate(data_types):
    data_type_df = df[df["data_type"] == data_type]

    for j, f1_col in enumerate(f1_columns):
        ax = axes[i, j] if len(data_types) > 1 else axes[j]

        # Get scores for normal and normalized models
        scores = []
        model_names = []

        for is_norm in [False, True]:
            filtered_data = data_type_df[data_type_df["is_normalized"] == is_norm]
            if not filtered_data.empty:
                score = filtered_data[f1_col].iloc[0]
                model_name = filtered_data["model_name"].iloc[0]
                scores.append(score)
                model_names.append(model_name)
            else:
                scores.append(0)
                model_names.append("N/A")

        # Create bars
        bars = ax.bar(
            x_positions,
            scores,
            bar_width,
            color=colors,
            alpha=0.8,
            edgecolor="black",
            linewidth=0.8,
        )

        # Add value labels on bars
        for bar, score in zip(bars, scores):
            if score > 0:
                height = bar.get_height()
                ax.text(
                    bar.get_x() + bar.get_width() / 2.0,
                    height + 0.02,
                    f"{score:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=12,
                    fontweight="bold",
                )

        # Customize subplot
        if i == 0:  # Only add column titles to top row
            ax.set_title(
                f"{f1_col.replace('_', ' ').title()}",
                fontweight="bold",
                pad=15,
                fontsize=14,
            )

        if j == 0:  # Only add row labels to first column
            ax.text(
                -0.15,
                0.5,
                data_type,
                rotation=90,
                va="center",
                ha="center",
                transform=ax.transAxes,
                fontweight="bold",
                fontsize=14,
            )

        if i == len(data_types) - 1:  # Only set x-ticks on bottom row
            ax.set_xticks(x_positions)
            ax.set_xticklabels(labels, fontsize=12)
        ax.set_ylabel("F1 Score" if j == 0 else "", fontsize=13, fontweight="bold")
        ax.set_ylim(0, 1.1)
        ax.grid(True, alpha=0.3, axis="y")

        # Clean up spines
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

# Add overall title and adjust layout
fig.suptitle(
    "Model Performance Comparison: Normal vs Normalized",
    fontsize=16,
    fontweight="bold",
    y=0.95,
)

plt.tight_layout()
plt.subplots_adjust(top=0.9, left=0.1)
plt.show()


## Training Curves

In [ ]:
models = [
    "ROSIDS23_blockcnn2d__nosampling_binary_20260111",
    # "ROSIDS23_blockcnn2d__normalized_nosampling_binary_20260111",
    "ROSIDS23_blockcnn2d__normalized_nosampling_binary_20260123",
    # "mobilenet_v3_large_nosampling_binary_20260121",
]
annotation_loc = [(75, 0.95), (75, 0.939)]
root = Path(r"rosaidresults")

plt.figure(figsize=(8, 5))
for model_name in models:
    dtype = "Normalized" if "normalized" in model_name else "Normal"
    dtype = "SPI" if model_name == "Normal" else "SPyI"
    df = pd.read_csv(root / "image_classification" / model_name / "metrics.csv")
    # Plot training and validation loss
    plt.plot(df["epoch"], df["f1_score"], label="Training Macro. F1 (" + dtype + ")")
    plt.plot(
        df["epoch"], df["val_f1_score"], label="Validation Macro. F1 (" + dtype + ")"
    )
    # annotate the best validation f1 score
    best_epoch = df["val_f1_score"].idxmax()
    best_f1 = df["val_f1_score"].max()
    plt.annotate(
        f"Best: {best_f1:.3f} \n(Epoch {best_epoch})",
        xy=(best_epoch, best_f1),
        xytext=annotation_loc[models.index(model_name)],
        arrowprops=dict(facecolor="black", arrowstyle="->"),
        fontsize=14,
        ha="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
    )
plt.legend(fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
# ticks font size
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel("Epoch", fontweight="bold", fontdict={"fontsize": 15})
plt.ylabel("Macro F1 Score", fontweight="bold", fontdict={"fontsize": 15})
plt.savefig("image_clf_curves.pdf", dpi=300, bbox_inches="tight")
plt.show()

### STFPM

In [ ]:
models = [
    "DNP3_blockcnn2d___ROSIDS23_nosampling_20260108",
    "blockcnn2d___ROSIDS23_nosampling_20260108",
]
annotation_loc = [(600, 5), (490, 5)]
root = Path(r"rosaidresults\image_stfpm")

plt.figure(figsize=(8, 5))
for model_name in models:
    df = pd.read_csv(root / model_name / "metrics.csv")
    model_kind = "DNP3" if "DNP3" in model_name else "ROSIDS23"
    # Plot training and validation loss
    plt.plot(df["epoch"], df["train_loss"], label=f"Training Loss({model_kind})")
    plt.plot(df["epoch"], df["val_loss"], label="Validation Loss(" + model_kind + ")")
    # annotate the best validation f1 score
    best_epoch = df["val_loss"].idxmin()
    best_loss = df["val_loss"].min()
    plt.annotate(
        f"{model_kind}\nBest: {best_loss:.3f} \n(Epoch {best_epoch})",
        xy=(best_epoch, best_loss),
        xytext=annotation_loc[models.index(model_name)],
        arrowprops=dict(facecolor="black", arrowstyle="->"),
        fontsize=14,
        ha="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
    )
plt.legend(fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
# ticks font size
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel("Epoch", fontweight="bold", fontdict={"fontsize": 15})
plt.ylabel("Loss", fontweight="bold", fontdict={"fontsize": 15})
plt.savefig("image_stfpm_curves.pdf", dpi=300, bbox_inches="tight")
plt.show()


## Visualize Errors

In [ ]:
eval_root = Path(
    r"rosaidresults\evaluation"
)
eval_csv = list(eval_root.rglob("*detailed*"))
print(f"Found {len(list(eval_csv))} detailed evaluation CSV files.")


In [ ]:
for csv_file in eval_csv:
    df = pd.read_csv(csv_file)
    if "anomaly_score" not in df.columns:
        continue
    print(f"Evaluating file: {csv_file.name} with {len(df)} entries.")
    # Further analysis can be done here
    # break
    df["is_clean"] = df["true_value"] == 0
    print(df.is_clean.value_counts())

    # box plot for each is_clean
    plt.figure(figsize=(8, 6))
    sns.boxplot(x="is_clean", y="anomaly_score", data=df)

    plt.show()

In [ ]:
df

In [ ]:
from matplotlib.ticker import FuncFormatter


def plot_histograms(df):
    clean_df = df.query("true_value==0").copy()
    attack_df = df.query("true_value!=0").copy()

    print(f"Clean df: {len(clean_df)}, Attack df: {len(attack_df)}")

    # Plot error distribution
    mse_columns = [c for c in df.columns if "mse" in c.lower()]
    ncols = len(mse_columns)

    # Determine layout: if we have many columns, split into 2 rows
    if ncols > 3:
        nrows = 2
        cols_per_row = (ncols + 1) // 2  # Ceiling division
    else:
        nrows = 1
        cols_per_row = ncols

    # Create figure with tighter spacing
    fig, axes = plt.subplots(nrows, cols_per_row, figsize=(3 * cols_per_row, 4 * nrows))

    # Handle different subplot configurations
    if nrows == 1:
        if cols_per_row == 1:
            axes = [axes]
        else:
            axes = [axes]
    else:
        if cols_per_row == 1:
            axes = axes.reshape(-1, 1)
        axes = axes.flatten()

    # Colors for clean vs attack
    clean_color = "#2E8B57"  # Sea Green
    attack_color = "#DC143C"  # Crimson

    for j, mse_col in enumerate(mse_columns):
        # Create refined column name
        if "block" in mse_col.lower():
            block_num = "".join(filter(str.isdigit, mse_col))
            refined_mse_col = f"Block {block_num}" if block_num else f"Block {j}"
        elif "layer" in mse_col.lower():
            layer_num = "".join(filter(str.isdigit, mse_col))
            refined_mse_col = f"Layer {layer_num}" if layer_num else f"Layer {j}"
        else:
            refined_mse_col = "Average MSE"

        ax = axes[j]

        # make mse col log scale
        ax.set_xscale("log")

        # Plot both distributions on same subplot
        sns.histplot(
            clean_df[mse_col],
            bins=40,
            # kde=True,
            color=clean_color,
            alpha=0.6,
            ax=ax,
            stat="density",
            label="Clean",
        )

        sns.histplot(
            attack_df[mse_col],
            bins=40,
            # kde=True,
            color=attack_color,
            alpha=0.6,
            ax=ax,
            stat="density",
            label="Attack",
        )

        # Determine current row and column
        if nrows == 1:
            current_row = 0
            current_col = j
        else:
            current_row = j // cols_per_row
            current_col = j % cols_per_row

        ax.set_title(f"{refined_mse_col}", fontsize=14, fontweight="bold", pad=10)

        # Only add x-label to bottom row
        if current_row == nrows - 1:
            ax.set_xlabel("MSE", fontsize=12, fontweight="bold")
        else:
            ax.set_xlabel("")

        # Only add y-label to leftmost column
        if current_col == 0:
            ax.set_ylabel("Density", fontsize=12, fontweight="bold")
        else:
            ax.set_ylabel("")

        # Improve tick visibility
        ax.tick_params(axis="both", which="major", labelsize=10, width=1.2, length=4)
        ax.tick_params(axis="both", which="minor", width=0.8, length=2)

        # Make ticks more visible with better formatting
        ax.xaxis.set_major_locator(plt.MaxNLocator(5))  # Limit to 5 major ticks
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))  # Limit to 4 major ticks

        ax.grid(True, alpha=0.3)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # Make remaining spines more visible
        ax.spines["left"].set_linewidth(1.2)
        ax.spines["bottom"].set_linewidth(1.2)

        # Add narrower legend
        legend = ax.legend(
            loc="upper right",
            fontsize=9,
            frameon=True,
            fancybox=False,  # Remove fancy box for narrower appearance
            shadow=False,  # Remove shadow for cleaner look
            handlelength=1.5,  # Shorter legend handles
            handletextpad=0.5,  # Less padding between handle and text
            columnspacing=0.5,  # Less space between columns if needed
            borderpad=0.3,  # Less padding inside legend box
        )

        # Make legend box narrower with custom styling
        legend.get_frame().set_linewidth(1.0)
        legend.get_frame().set_edgecolor("gray")
        legend.get_frame().set_facecolor("white")
        legend.get_frame().set_alpha(0.9)

        # Calculate statistics with scientific notation formatting
        clean_mean = clean_df[mse_col].mean()
        clean_std = clean_df[mse_col].std()
        attack_mean = attack_df[mse_col].mean()
        attack_std = attack_df[mse_col].std()

        # Format numbers in scientific notation if they're very large or very small
        def format_sci(num):
            if abs(num) >= 1000 or abs(num) <= 0.001:
                return f"{num:.2e}"
            else:
                return f"{num:.4f}"

        # Add clean statistics text on the left with matching color
        clean_stats_text = (
            f"Clean\nμ={format_sci(clean_mean)}\nσ={format_sci(clean_std)}"
        )
        ax.text(
            0.02,
            0.75,  # Moved down slightly to avoid overlap with legend
            clean_stats_text,
            transform=ax.transAxes,
            fontsize=9,
            verticalalignment="top",
            horizontalalignment="left",
            color=clean_color,
            fontweight="bold",
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                alpha=0.9,
                edgecolor=clean_color,
                linewidth=1.2,
            ),
        )

        # Add attack statistics text on the left below clean stats
        attack_stats_text = (
            f"Attack\nμ={format_sci(attack_mean)}\nσ={format_sci(attack_std)}"
        )
        ax.text(
            0.02,
            0.45,  # Position below clean stats
            attack_stats_text,
            transform=ax.transAxes,
            fontsize=9,
            verticalalignment="top",
            horizontalalignment="left",
            color=attack_color,
            fontweight="bold",
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                alpha=0.9,
                edgecolor=attack_color,
                linewidth=1.2,
            ),
        )

        def y_tick_formatter(x, pos):
            if abs(x) >= 1000 or (abs(x) <= 0.001 and x != 0):
                # Convert to scientific notation and format as 1eX
                exp_str = f"{x:.1e}"
                coeff, exp = exp_str.split("e")
                exp = int(exp)
                return f"{float(coeff):.1f}e{exp}"
            else:
                return f"{x:.2f}"

        ax.yaxis.set_major_formatter(FuncFormatter(y_tick_formatter))

    # Hide extra subplots if ncols is odd and we have 2 rows
    if nrows == 2 and ncols % 2 == 1:
        axes[-1].set_visible(False)

    # Adjust layout with tighter spacing
    plt.tight_layout()
    plt.subplots_adjust(top=0.92, hspace=0.25, wspace=0.2)
    # plt.show()
    return fig, axes


for csv_file in eval_csv:
    if "104" not in csv_file.name:
        continue
    odf = pd.read_csv(csv_file)
    print(f"Evaluating file: {csv_file.name} with {len(df)} entries.")

    unique_values = odf["true_value"].unique()
    print(f"Unique true_value entries: {unique_values}")
    for uv in unique_values:
        if uv == 0:
            continue
        # uv==0 and uv!=0
        labels = ["BENIGN", "DOS", "SUBFLOOD", "UNAUTHPUB", "UNAUTHSUB"]
        print(f"Analyzing true_value: BENIGN and {labels[uv]}")
        df = odf.query(f"true_value=={uv} or true_value==0").copy()

        fig, axes = plot_histograms(df)
        fig.savefig(
            f"{csv_file.parent}/histograms.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()
        # based on avg_mse, test if clean/attack have different distributions
        # use scipy.stats.ks_2samp
        # based on avg_mse, test if clean/attack have different distributions
        # use different statistical tests

        from scipy.stats import ks_2samp
        from scipy.stats import mannwhitneyu, normaltest, shapiro, levene

        clean_df = df.query("true_value==0").copy()
        attack_df = df.query("true_value!=0").copy()
        clean_mse = clean_df["avg_mse"].values
        attack_mse = attack_df["avg_mse"].values

        # compute tests
        ks_stat, p_value = ks_2samp(clean_mse, attack_mse)
        mannwhitneyu_stat, mannwhitneyu_p = mannwhitneyu(
            clean_mse, attack_mse, alternative="two-sided"
        )
        normaltest_stat, normaltest_p = normaltest(clean_mse)
        shapiro_stat, shapiro_p = shapiro(clean_mse)
        levene_stat, levene_p = levene(clean_mse, attack_mse)
        print(f"KS Statistic: {ks_stat}, P-value: {p_value}")
        print(
            f"Mann-Whitney U Statistic: {mannwhitneyu_stat}, P-value: {mannwhitneyu_p}"
        )
        print(f"Normality Test Statistic: {normaltest_stat}, P-value: {normaltest_p}")
        print(f"Shapiro-Wilk Test Statistic: {shapiro_stat}, P-value: {shapiro_p}")
        print(f"Levene's Test Statistic: {levene_stat}, P-value: {levene_p}")

## OOD Plots

In [ ]:
import numpy as np

# clf model's result
fdia_result_root = Path(
    r"rosaidresults\evaluation\OOD_evaluation"
)

for model_dir in fdia_result_root.iterdir():
    pass


In [ ]:
import numpy as np


fdia_result_root = Path(
    r"rosaidresults\evaluation\OOD_evaluation"
)

thresholds = {
    "DNP3": {
        "percentile": 88,
        "threshold_value": 1.8505119027593723e-19,
        "threshold_value_log10": -18.73270811711516,
        "best_f1": 0.862697953575659,
    },
    "ROSIDS23": {
        "percentile": 69,
        "threshold_value": 1.1024399467585079e-20,
        "threshold_value_log10": -19.95764505855104,
        "best_f1": 0.7572698061198747,
    },
}
stfpm_models = list(fdia_result_root.iterdir())


# Calculate 97th percentile of benign scores for ROSIDS23 once
first_model = stfpm_models[0]
first_data_type = list(first_model.iterdir())[0]
baseline_feat_df = pd.read_csv(first_data_type / "stfpm_results.csv")
baseline_feat_df["anomaly_score"] = np.log10(baseline_feat_df["anomaly_score"])
rosids23_data = None


for stfpm_model in stfpm_models:
    for data_type in stfpm_model.iterdir():
        if "ROSIDS23" in data_type.name:
            stfpm_df = pd.read_csv(data_type / "stfpm_results.csv")
            stfpm_df["anomaly_score"] = np.log10(stfpm_df["anomaly_score"])
            rosids23_benign_scores = stfpm_df.query("true_binary_label!='ATTACK'")[
                "anomaly_score"
            ].values
            rosids23_data = rosids23_benign_scores
            break


threshold_97 = np.percentile(rosids23_data, 97) if rosids23_data is not None else None
print(f"97th percentile of ROSIDS23 benign scores (global): {threshold_97:.4f}")


# Create subplots: len(stfpm_models) rows, 1 column
n_rows = len(stfpm_models)
fig, axes = plt.subplots(n_rows, 1, figsize=(8, 8), sharey=False)


if n_rows == 1:
    axes = [axes]


for row_idx, stfpm_model in enumerate(stfpm_models):
    final_res = {}

    for data_type in stfpm_model.iterdir():
        stfpm_df = pd.read_csv(data_type / "stfpm_results.csv")
        print(
            f"STFPM Model: {stfpm_model.name}, Data Type: {data_type.name}, Entries: {len(stfpm_df)}"
        )
        # log10 the anomaly_score
        stfpm_df["anomaly_score"] = np.log10(stfpm_df["anomaly_score"])
        atk_scores = stfpm_df.query("true_binary_label=='ATTACK'")[
            "anomaly_score"
        ].values
        benign_scores = stfpm_df.query("true_binary_label!='ATTACK'")[
            "anomaly_score"
        ].values
        final_res[data_type.name] = {
            "atk_scores": atk_scores,
            "benign_scores": benign_scores,
        }

    # plot violin plot of final_res in one row len(data_types) columns
    # each subplot has two violin plots: atk_scores and benign_scores
    ncols = len(final_res)

    # Create inner subplots for this model
    ax_main = axes[row_idx]

    # Plot all data types in one row within this main subplot
    x_offset = 0
    for col_idx, (data_type, scores) in enumerate(final_res.items()):
        if data_type == "ROSIDS23":
            continue

        box_data = [scores["benign_scores"], scores["atk_scores"]]
        parts = ax_main.violinplot(
            box_data,
            positions=[x_offset + 1, x_offset + 2],
            widths=0.7,
            showmeans=False,
            showmedians=True,
            showextrema=True,
        )

        # Color violins
        for pc in parts["bodies"]:
            pc.set_facecolor("#ADD8E6")
            pc.set_alpha(0.7)
            pc.set_edgecolor("black")
            pc.set_linewidth(1.5)

        # Style median and other components
        parts["cmedians"].set_edgecolor("red")
        parts["cmedians"].set_linewidth(2)
        parts["cbars"].set_edgecolor("black")
        parts["cmaxes"].set_edgecolor("black")
        parts["cmins"].set_edgecolor("black")

        # Calculate accuracy for each violin
        benign_scores = scores["benign_scores"]
        atk_scores = scores["atk_scores"]

        # For benign: should be below threshold (for ROSIDS23) or any for others
        if data_type == "ROSIDS23":
            # Benign should be below threshold
            benign_correct = np.sum(benign_scores < threshold_97)
            benign_accuracy = (
                benign_correct / len(benign_scores) if len(benign_scores) > 0 else 0
            )
        else:
            # All should be above threshold
            benign_correct = np.sum(benign_scores > threshold_97)
            benign_accuracy = (
                benign_correct / len(benign_scores) if len(benign_scores) > 0 else 0
            )

        # For attack: should be above threshold
        atk_correct = np.sum(atk_scores > threshold_97)
        atk_accuracy = atk_correct / len(atk_scores) if len(atk_scores) > 0 else 0

        # Calculate average accuracy
        avg_accuracy = (benign_accuracy + atk_accuracy) / 2

        # Get y-axis limits for text positioning in center
        y_min, y_max = ax_main.get_ylim()
        y_center = (y_min + y_max) / 2

        # Add textbox with accuracy in center of each violin
        props_benign = dict(
            boxstyle="round,pad=0.3", facecolor="#ADD8E6", edgecolor="blue", linewidth=2
        )
        props_attack = dict(
            boxstyle="round,pad=0.3", facecolor="#FFB6C6", edgecolor="red", linewidth=2
        )

        ax_main.text(
            x_offset + 1,
            y_center,
            f"{benign_accuracy:.3f}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            bbox=props_benign,
        )
        ax_main.text(
            x_offset + 2,
            y_center,
            f"{atk_accuracy:.3f}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            bbox=props_attack,
        )

        # Add labels below violins (Benign and Attack)
        ax_main.text(
            x_offset + 1,
            y_min + 0.3,
            "Benign",
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color="darkblue",
        )
        ax_main.text(
            x_offset + 2,
            y_min + 0.3,
            "Attack",
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color="darkred",
        )

        # Add label above this data type
        ax_main.text(
            x_offset + 1.5,
            y_max + 0.5,
            f"{data_type}\n(Avg: {avg_accuracy:.3f})",
            ha="center",
            va="bottom",
            fontsize=13,
            fontweight="bold",
        )

        x_offset += 3

    # Draw 97th percentile threshold line
    if threshold_97 is not None:
        ax_main.axhline(
            y=threshold_97, color="red", linestyle=":", linewidth=2.5, alpha=0.8
        )

    ax_main.set_ylabel("Anomaly Score (Log10)", fontsize=13, fontweight="bold")
    model_name = "DNP3 Teacher" if "DNP3" in stfpm_model.name else "ROSIDS23 Teacher"
    ax_main.set_title(f"{model_name}", fontsize=14, fontweight="bold", pad=10)
    ax_main.grid(True, alpha=0.3)
    ax_main.tick_params(axis="y", labelsize=12)

    # Set y-axis limits with extra space for annotations
    current_y_min = ax_main.get_ylim()[0]
    current_y_max = ax_main.get_ylim()[1] + 0.5
    y_range = current_y_max - current_y_min
    ax_main.set_ylim(current_y_min, current_y_max + y_range * 0.15)

    # Remove x-axis ticks as they're not meaningful across multiple data types
    ax_main.set_xticks([])

    if row_idx == 0:
        # Add legend only to first subplot
        from matplotlib.lines import Line2D

        legend_elements = [
            Line2D(
                [0],
                [0],
                color="red",
                linestyle=":",
                linewidth=2.5,
                label="97th pctl (ROSIDS23)",
            )
        ]
        # ax_main.legend(handles=legend_elements, fontsize=11, loc="best")


plt.tight_layout()
plt.savefig("ood_all_models.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score


fdia_result_root = Path(
    r"rosaidresults\evaluation\OOD_evaluation"
)


thresholds = {
    "DNP3": {
        "percentile": 88,
        "threshold_value": 1.8505119027593723e-19,
        "threshold_value_log10": -18.73270811711516,
        "best_f1": 0.862697953575659,
        "accuracy": 0.8682727079344821,
    },
    "ROSIDS23": {
        "percentile": 69,
        "threshold_value": 1.1024399467585079e-20,
        "threshold_value_log10": -19.95764505855104,
        "best_f1": 0.7572698061198747,
        "accuracy": 0.7723888534354393,
    },
}
stfpm_models = list(fdia_result_root.iterdir())


violin_colors = ["#ADD8E6", "#ADD8E6"]  # Light blue for DNP3, light orange for ROSIDS23
colors_line = ["blue", "blue"]  # Line colors for DNP3 and ROSIDS23


# Create subplots: len(stfpm_models) rows, 1 column
n_rows = len(stfpm_models)
fig, axes = plt.subplots(n_rows, 1, figsize=(8, 10), sharey=False)


if n_rows == 1:
    axes = [axes]


for row_idx, stfpm_model in enumerate(stfpm_models):
    final_res = {}

    # Determine teacher name and color index
    is_dnp3 = "DNP3" in stfpm_model.name
    teacher_name = "DNP3" if is_dnp3 else "ROSIDS23"
    color_idx = 0 if is_dnp3 else 1
    violin_color = violin_colors[color_idx]
    line_color = colors_line[color_idx]
    thresh_acc = {}
    for data_type in stfpm_model.iterdir():
        # Get threshold for this model
        threshold_log10 = thresholds[teacher_name]["threshold_value_log10"]
        best_acc = thresholds[teacher_name]["accuracy"]
        percentile = thresholds[teacher_name]["percentile"]
        stfpm_df = pd.read_csv(data_type / "stfpm_results.csv")
        # stfpm_df = stfpm_df.drop_duplicates(subset=["image_name"])
        print(
            f"STFPM Model: {stfpm_model.name}, Data Type: {data_type.name}, Entries: {len(stfpm_df)}"
        )
        # log10 the anomaly_score
        stfpm_df["anomaly_score"] = np.log10(stfpm_df["anomaly_score"])
        atk_scores = stfpm_df.query("true_binary_label=='ATTACK'")[
            "anomaly_score"
        ].values
        benign_scores = stfpm_df.query("true_binary_label!='ATTACK'")[
            "anomaly_score"
        ].values
        final_res[data_type.name] = {
            "atk_scores": atk_scores,
            "benign_scores": benign_scores,
        }
        if not data_type.name == "ROSIDS23":
            true_label = np.ones(len(stfpm_df))
            pred_label = stfpm_df["anomaly_score"] > threshold_log10

            best_acc = accuracy_score(true_label, pred_label)
            print(
                f"F1 Score for {data_type.name} with threshold at {threshold_log10:.4f}: {best_acc:.4f}"
            )
        thresh_acc[data_type.name] = best_acc

    # Create inner subplots for this model
    ax_main = axes[row_idx]

    # Plot all data types in one row within this main subplot
    x_offset = 0

    for col_idx, (data_type, scores) in enumerate(final_res.items()):
        if data_type == "ROSIDS23":
            continue
        box_data = [scores["benign_scores"], scores["atk_scores"]]
        parts = ax_main.violinplot(
            box_data,
            positions=[x_offset + 1, x_offset + 1.8],
            widths=0.7,
            showmeans=False,
            showmedians=False,
            showextrema=True,
        )

        # Color violins with matching color
        for pc in parts["bodies"]:
            pc.set_facecolor(violin_color)
            pc.set_alpha(0.7)
            pc.set_edgecolor("black")
            pc.set_linewidth(1.5)

        # Style median and other components
        # parts["cmedians"].set_edgecolor("red")
        # parts["cmedians"].set_linewidth(2.5)
        parts["cbars"].set_edgecolor("black")
        parts["cmaxes"].set_edgecolor("black")
        parts["cmins"].set_edgecolor("black")

        # Get y-axis limits for text positioning
        y_min, y_max = ax_main.get_ylim()

        # Add labels below violins (Benign and Attack)
        ax_main.text(
            x_offset + 1,
            y_min + 0.1,
            "BENIGN",
            ha="center",
            va="center",
            fontsize=16,
            fontweight="bold",
            # color="darkblue",
        )
        ax_main.text(
            x_offset + 1.8,
            y_min + 0.1,
            "ATTACK",
            ha="center",
            va="center",
            fontsize=16,
            fontweight="bold",
            # color="darkred",
        )

        # Add label above this data type with Accuracy
        ax_main.text(
            x_offset + 1.25,
            y_max * 0.98,
            f"Data: {data_type}\nAcc: {thresh_acc[data_type]:.3f}",
            ha="center",
            va="top",
            fontsize=15,
            fontweight="bold",
            bbox=dict(
                boxstyle="round,pad=0.7",
                facecolor=violin_color,
                edgecolor="black",
                linewidth=2.5,
                alpha=0.9,
            ),
        )

        x_offset += 2

    # Draw threshold line with matching color
    if threshold_log10 is not None:
        ax_main.plot(
            [0.5, x_offset + 0.25],
            [threshold_log10, threshold_log10],
            color=line_color,
            linestyle=":",
            linewidth=4,
            alpha=0.8,
        )
    avg_accuracy = np.mean(list(thresh_acc.values()))
    ax_main.set_ylabel("Anomaly Score (Log10)", fontsize=18, fontweight="bold")
    model_name = f"{teacher_name} Teacher (Acc: {avg_accuracy:.3f})"
    ax_main.set_title(f"{model_name}", fontsize=19, fontweight="bold", pad=25)
    ax_main.grid(True, alpha=0.6, linestyle="--", linewidth=1.3)
    ax_main.tick_params(axis="y", labelsize=15)
    ax_main.legend(
        handles=[
            Line2D(
                [0],
                [0],
                color=line_color,
                linestyle=":",
                linewidth=4,
                label="Threshold",
            )
        ],
        fontsize=14,
        loc="upper center",
    )

    # Set y-axis limits with extra space for annotations
    current_y_min = ax_main.get_ylim()[0]
    current_y_max = ax_main.get_ylim()[1]
    y_range = current_y_max - current_y_min
    ax_main.set_ylim(current_y_min - y_range * 0.05, current_y_max + y_range * 0.15)

    # Remove x-axis ticks as they're not meaningful across multiple data types
    ax_main.set_xticks([])

    # Increase spine width for academic publishing
    for spine in ax_main.spines.values():
        spine.set_linewidth(2.5)


plt.tight_layout()
plt.savefig("ood_all_models.pdf", dpi=300, bbox_inches="tight")
plt.show()


#### With Best

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score


fdia_result_root = Path(
    r"rosaidresults\evaluation\OOD_evaluation"
)


thresholds = {
    "DNP3": {
        "percentile": 88,
        "threshold_value": 1.8505119027593723e-19,
        "threshold_value_log10": -18.73270811711516,
        "best_f1": 0.862697953575659,
        "accuracy": 0.8682727079344821,
    },
    "ROSIDS23": {
        "percentile": 69,
        "threshold_value": 1.1024399467585079e-20,
        "threshold_value_log10": -19.95764505855104,
        "best_f1": 0.7572698061198747,
        "accuracy": 0.7723888534354393,
    },
}
stfpm_models = list(fdia_result_root.iterdir())


violin_colors = ["#ADD8E6", "#FFE4B5"]  # Light blue for DNP3, light orange for ROSIDS23
colors_line = ["blue", "green"]  # Line colors for DNP3 and ROSIDS23


# Create subplots: len(stfpm_models) rows, 1 column
n_rows = len(stfpm_models) - 1
fig, axes = plt.subplots(n_rows, 1, figsize=(8, 5), sharey=False)


if n_rows == 1:
    axes = [axes]


for row_idx, stfpm_model in enumerate(stfpm_models):
    final_res = {}

    # Determine teacher name and color index
    is_dnp3 = "DNP3" in stfpm_model.name
    teacher_name = "DNP3" if is_dnp3 else "ROSIDS23"
    if teacher_name == "ROSIDS23":
        continue
    color_idx = 0 if is_dnp3 else 1
    violin_color = violin_colors[color_idx]
    line_color = colors_line[color_idx]
    thresh_acc = {}
    for data_type in stfpm_model.iterdir():
        # Get threshold for this model
        threshold_log10 = thresholds[teacher_name]["threshold_value_log10"]
        best_acc = thresholds[teacher_name]["accuracy"]
        percentile = thresholds[teacher_name]["percentile"]
        stfpm_df = pd.read_csv(data_type / "stfpm_results.csv")
        # stfpm_df = stfpm_df.drop_duplicates(subset=["image_name"])
        print(
            f"STFPM Model: {stfpm_model.name}, Data Type: {data_type.name}, Entries: {len(stfpm_df)}"
        )
        # log10 the anomaly_score
        stfpm_df["anomaly_score"] = np.log10(stfpm_df["anomaly_score"])
        atk_scores = stfpm_df.query("true_binary_label=='ATTACK'")[
            "anomaly_score"
        ].values
        benign_scores = stfpm_df.query("true_binary_label!='ATTACK'")[
            "anomaly_score"
        ].values
        final_res[data_type.name] = {
            "atk_scores": atk_scores,
            "benign_scores": benign_scores,
        }
        if not data_type.name == "ROSIDS23":
            true_label = np.ones(len(stfpm_df))
            pred_label = stfpm_df["anomaly_score"] > threshold_log10

            best_acc = accuracy_score(true_label, pred_label)
            print(
                f"F1 Score for {data_type.name} with threshold at {threshold_log10:.4f}: {best_acc:.4f}"
            )
        thresh_acc[data_type.name] = best_acc

    # Create inner subplots for this model
    ax_main = axes[0]

    # Plot all data types in one row within this main subplot
    x_offset = 0

    for col_idx, (data_type, scores) in enumerate(final_res.items()):
        if data_type == "ROSIDS23":
            continue
        box_data = [scores["benign_scores"], scores["atk_scores"]]
        parts = ax_main.violinplot(
            box_data,
            positions=[x_offset + 1, x_offset + 1.8],
            widths=0.7,
            showmeans=False,
            showmedians=False,
            showextrema=True,
        )

        # Color violins with matching color
        for pc in parts["bodies"]:
            pc.set_facecolor(violin_color)
            pc.set_alpha(0.7)
            pc.set_edgecolor("black")
            pc.set_linewidth(1.5)

        # Style median and other components
        # parts["cmedians"].set_edgecolor("red")
        # parts["cmedians"].set_linewidth(2.5)
        parts["cbars"].set_edgecolor("black")
        parts["cmaxes"].set_edgecolor("black")
        parts["cmins"].set_edgecolor("black")

        # Get y-axis limits for text positioning
        y_min, y_max = ax_main.get_ylim()

        # Add labels below violins (Benign and Attack)
        ax_main.text(
            x_offset + 1,
            y_min + 0.1,
            "BENIGN",
            ha="center",
            va="center",
            fontsize=16,
            fontweight="bold",
            color="darkblue",
        )
        ax_main.text(
            x_offset + 1.8,
            y_min + 0.1,
            "ATTACK",
            ha="center",
            va="center",
            fontsize=16,
            fontweight="bold",
            color="darkred",
        )

        # Add label above this data type with Accuracy
        ax_main.text(
            x_offset + 1.25,
            y_max * 0.98,
            f"{data_type}\nAcc: {thresh_acc[data_type]:.3f}",
            ha="center",
            va="top",
            fontsize=15,
            fontweight="bold",
            bbox=dict(
                boxstyle="round,pad=0.7",
                facecolor=violin_color,
                edgecolor="black",
                linewidth=2.5,
                alpha=0.9,
            ),
        )

        x_offset += 2

    # Draw threshold line with matching color
    if threshold_log10 is not None:
        ax_main.plot(
            [0.5, x_offset + 0.25],
            [threshold_log10, threshold_log10],
            color=line_color,
            linestyle=":",
            linewidth=4,
            alpha=0.8,
        )
    avg_accuracy = np.mean(list(thresh_acc.values()))
    ax_main.set_ylabel("Anomaly Score (Log10)", fontsize=18, fontweight="bold")
    model_name = f"{teacher_name} Teacher (Acc: {avg_accuracy:.3f})"
    ax_main.set_title(f"{model_name}", fontsize=19, fontweight="bold", pad=25)
    ax_main.grid(True, alpha=0.6, linestyle="--", linewidth=1.3)
    ax_main.tick_params(axis="y", labelsize=15)
    ax_main.legend(
        handles=[
            Line2D(
                [0],
                [0],
                color=line_color,
                linestyle=":",
                linewidth=4,
                label=f"Threshold: {threshold_log10:.3f}",
            )
        ],
        fontsize=14,
        loc="upper right",
    )

    # Set y-axis limits with extra space for annotations
    current_y_min = ax_main.get_ylim()[0]
    current_y_max = ax_main.get_ylim()[1]
    y_range = current_y_max - current_y_min
    ax_main.set_ylim(current_y_min - y_range * 0.05, current_y_max + y_range * 0.15)

    # Remove x-axis ticks as they're not meaningful across multiple data types
    ax_main.set_xticks([])

    # Increase spine width for academic publishing
    for spine in ax_main.spines.values():
        spine.set_linewidth(2.5)


plt.tight_layout()
plt.savefig("ood_all_models.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Academic model colors
model_colors = {
    "DNP3": {"line": "#0057B7", "fill": "#aec7e8"},
    "ROSIDS23": {"line": "#A52A2A", "fill": "#ffbb78"},
}

n_rows = len(stfpm_models) - 1
fig, axes = plt.subplots(n_rows, 1, figsize=(8, 5), sharey=False)

if n_rows == 1:
    axes = [axes]

for row_idx, stfpm_model in enumerate(stfpm_models):
    teacher_name = "DNP3" if "DNP3" in stfpm_model.name else "ROSIDS23"
    if teacher_name == "ROSIDS23":
        continue

    line_color = model_colors[teacher_name]["line"]
    violin_color = model_colors[teacher_name]["fill"]

    ax_main = axes[0]
    x_offset = 0

    # Lists to store axis tick info
    tick_positions = []
    tick_labels = []

    for data_type, scores in final_res.items():
        if data_type == "ROSIDS23":
            continue

        pos_benign = x_offset + 1
        pos_attack = x_offset + 1.7
        box_data = [scores["benign_scores"], scores["atk_scores"]]

        parts = ax_main.violinplot(
            box_data, positions=[pos_benign, pos_attack], widths=0.6
        )

        # Body styling
        for pc in parts["bodies"]:
            pc.set_facecolor(violin_color)
            pc.set_alpha(0.7)
            pc.set_edgecolor("black")
            pc.set_linewidth(1.5)

        # Extrema styling
        for part in ["cbars", "cmaxes", "cmins"]:
            parts[part].set_edgecolor("black")
            parts[part].set_linewidth(1.5)

        # Add to tick lists
        tick_positions.extend([pos_benign, pos_attack])
        tick_labels.extend(["BENIGN", "ATTACK"])

        # Data Type Annotation (Box above plot)
        y_max = ax_main.get_ylim()[1]
        ax_main.text(
            x_offset + 1.14,
            -7,
            f"Data:{data_type}\nAcc: {thresh_acc[data_type]:.3f}",
            ha="center",
            va="top",
            fontsize=13,
            fontweight="bold",
            bbox=dict(
                boxstyle="round,pad=0.4",
                facecolor=violin_color,
                edgecolor="black",
                linewidth=1.5,
            ),
        )

        x_offset += 1.7

    # Draw Threshold Line
    ax_main.axhline(
        threshold_log10, color=line_color, linestyle=":", linewidth=3, alpha=0.8
    )

    # SET X-AXIS LABELS (Black and Bold)
    ax_main.set_xticks(tick_positions)
    ax_main.set_xticklabels(tick_labels, fontsize=14, fontweight="bold", color="black")

    # yticks
    ax_main.tick_params(axis="y", labelsize=14, color="black", width=1.5)

    # Label styling
    ax_main.set_ylabel("Anomaly Score (Log₁₀)", fontsize=16, fontweight="bold")
    ax_main.set_title(
        f"{teacher_name} Teacher (Avg Acc: {np.mean(list(thresh_acc.values())):.3f})",
        fontsize=17,
        fontweight="bold",
        pad=20,
    )
    ax_main.grid(True, alpha=0.4, linestyle="--")

    # Legend
    legend_elements = [
        Line2D(
            [0],
            [0],
            color=line_color,
            linestyle=":",
            linewidth=3,
            label=f"Threshold: {threshold_log10:.2f}",
        )
    ]
    ax_main.legend(handles=legend_elements, loc="upper right", fontsize=12)

    # Spines
    for spine in ax_main.spines.values():
        spine.set_linewidth(2)

plt.tight_layout()
plt.savefig("ood_all_models.pdf", dpi=300, bbox_inches="tight")
plt.show()


### OOD on Baseline Model

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix


for row_idx, stfpm_model in enumerate(stfpm_models):
    final_res = {}
    print(f"Processing STFPM Model: {stfpm_model}")
    for data_type in stfpm_model.iterdir():
        if "ROS" in data_type.name:
            continue
        clf_df = pd.read_csv(data_type / "classifier_results.csv")
        true_labels = clf_df["true_binary_label"].values
        pred_labels = clf_df["clf_pred_clean"].values
        # set all true labels to "ATTACK" bcz its ood
        true_labels = ["ATTACK"] * len(true_labels)

        # compute accuracy
        accuracy = accuracy_score(true_labels, pred_labels)
        print(f"Data Type: {data_type.name}, Classifier Accuracy: {accuracy:.4f}")

        cm = confusion_matrix(true_labels, pred_labels)
        print(f"Confusion Matrix:\n{cm}\n")

        print(clf_df.groupby("true_binary_label")["clf_pred_clean"].value_counts())

    break

In [ ]:
clf_df.clf_pred_clean.value_counts()


## Saliency Maps

In [ ]:
import torch
import cv2


normal = cv2.imread(
    r"rosaidassets\figures\subcriberflood.png"
)[:100]
normalized_img = cv2.imread(
    r"rosaidassets\figures\subcriberflood_normalized.png"
)[:100]

# both to grayscale
normal = cv2.cvtColor(normal, cv2.COLOR_BGR2GRAY)
normalized_img = cv2.cvtColor(normalized_img, cv2.COLOR_BGR2GRAY)

# if normal width is not 256, pad it
if normal.shape[1] < 256:
    pad_width = 256 - normal.shape[1]
    normal = cv2.copyMakeBorder(
        normal, 0, 0, 0, pad_width, cv2.BORDER_CONSTANT, value=0
    )

normal_model = torch.load(
    r"rosaidresults\image_classification\ROSIDS23_blockcnn2d__nosampling_binary_20260111\best_model_full.pth",
    weights_only=False,
    map_location="cpu",
)
normalized_model = torch.load(
    r"rosaidresults\image_classification\ROSIDS23_blockcnn2d__normalized_nosampling_binary_20260111\best_model_full.pth",
    weights_only=False,
    map_location="cpu",
)


In [ ]:
import numpy as np
import torch
import numpy as np


class SaliencyMap:
    def __init__(self, model):
        self.model = model
        self.model.eval()

    def generate_saliency(self, input_image: np.ndarray, target_class=None):
        """
        Generate saliency map for binary classification model with single output

        Args:
            input_image: numpy array of shape (H, W)
            target_class: not used for binary classification
        """
        # Convert to torch tensor and normalize if needed
        if input_image.dtype == np.uint8:
            input_tensor = torch.from_numpy(input_image).float() / 255.0
        else:
            input_tensor = torch.from_numpy(input_image).float()

        # Add batch and channel dimensions: (H, W) -> (1, 1, H, W)
        input_tensor = input_tensor.unsqueeze(0).unsqueeze(0)

        # Enable gradients
        input_tensor.requires_grad_(True)

        print(f"Input tensor shape: {input_tensor.shape}")

        # Forward pass
        self.model.zero_grad()
        output = self.model(input_tensor)

        # Handle different output formats
        if isinstance(output, tuple):
            # If your model returns (logits, prob)
            logits, prob = output
            target_score = prob.squeeze()  # Remove batch dimension
        else:
            # If your model returns single value
            target_score = output.squeeze()  # Remove batch dimension

        print(f"Model output: {target_score.item():.4f}")

        # Backward pass - use the output directly for binary classification
        target_score.backward()

        # Get gradients and create saliency map
        gradients = input_tensor.grad.data.abs()

        # Remove batch and channel dimensions: (1, 1, H, W) -> (H, W)
        saliency = gradients.squeeze().numpy()
        print(f"Max max saliency value: {saliency.max():.4f}")
        # Normalize to [0, 1]
        if saliency.max() > saliency.min():
            saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min())
        else:
            saliency = np.zeros_like(saliency)

        return saliency

    def generate_integrated_gradients(self, input_image: np.ndarray, steps=50):
        """
        Generate Integrated Gradients saliency map for better attribution
        """
        # Convert input
        if input_image.dtype == np.uint8:
            input_tensor = torch.from_numpy(input_image).float() / 255.0
        else:
            input_tensor = torch.from_numpy(input_image).float()

        input_tensor = input_tensor.unsqueeze(0).unsqueeze(0)

        # Create baseline (zeros)
        baseline = torch.zeros_like(input_tensor)

        # Generate path from baseline to input
        alphas = torch.linspace(0, 1, steps)
        integrated_gradients = torch.zeros_like(input_tensor)

        for alpha in alphas:
            # Interpolated input
            interpolated_input = baseline + alpha * (input_tensor - baseline)
            interpolated_input.requires_grad_(True)

            # Forward pass
            self.model.zero_grad()
            output = self.model(interpolated_input)

            if isinstance(output, tuple):
                logits, prob = output
                target_score = prob.squeeze()
            else:
                target_score = output.squeeze()

            # Backward pass
            target_score.backward()

            # Accumulate gradients
            integrated_gradients += interpolated_input.grad

        # Average gradients and multiply by input difference
        integrated_gradients /= steps
        integrated_gradients *= input_tensor - baseline

        # Process gradients
        gradients = integrated_gradients.abs()
        saliency = gradients.squeeze().numpy()

        # Normalize
        if saliency.max() > saliency.min():
            saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min())
        else:
            saliency = np.zeros_like(saliency)

        return saliency


normal_saliency_generator = SaliencyMap(normal_model)
normalized_saliency_generator = SaliencyMap(normalized_model)

# plot 2,2 subplot: normal image, normal saliency, normalized image, normalized saliency
fig, axes = plt.subplots(2, 2, figsize=(8, 4))
# Normal image and saliency
axes[0, 0].imshow(normal, cmap="gray")
normal_saliency = normal_saliency_generator.generate_saliency(normal)
axes[1, 0].imshow(normal_saliency, cmap="jet")
# Normalized image and saliency
axes[0, 1].imshow(normalized_img, cmap="gray")
normalized_saliency = normalized_saliency_generator.generate_saliency(normalized_img)
axes[1, 1].imshow(normalized_saliency, cmap="jet")
# Set titles
axes[0, 0].set_title("Normal", fontsize=14, fontweight="bold")
axes[0, 0].set_ylabel("Image", fontsize=13, fontweight="bold")
# axes[1, 0].set_title("Normal Image Saliency Map", fontsize=14, fontweight="bold")
axes[0, 1].set_title("Normalised", fontsize=14, fontweight="bold")
axes[1, 0].set_ylabel("Saliency Map", fontsize=13, fontweight="bold")
# axes[1, 1].set_title("Saliency Map", fontsize=14, fontweight="bold")
# Remove ticks but show the y labels for left column
for i in range(2):
    for j in range(2):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])


plt.tight_layout()
plt.savefig(
    "saliency_comparison.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
normal_saliency.max(), normalized_saliency.max()